In [1]:
import pandas as pd
import re
import matplotlib.pyplot as plt
import os
from ast import literal_eval
from collections import Counter
import math
import numpy as np
import seaborn as sns

In [2]:
username = os.path.expanduser('~')
username

'/home/ehdgus7279'

In [3]:
benchmarks = [
    # Low-sharing Apps
    "fdtd2d",
    "jacobi1d",
    "fastwalshtransform",
    "relu",
    "fft",
    "fir",
    "sssp",
    "convolution2d",
    "color",
    "bitonicsort",
    "pagerank",
    "simpleconvolution",

    # High-sharing Apps
    'gups',    
    'syr2k',
    'spmv',
    'mis', 
    'stencil2d',
    'gemm',
    'lu',
    'matrixmultiplication',
    'mm2',
    'mm3',
    'gaussian',
    'matrixtranspose',
    'im2col',
    'vgg16',
    'bfs',
]

In [4]:
num_chiplets = 36
ns = 0.000000001
chip_cols = [str(i) for i in range(36)]

In [5]:
base_dir=f'{username}/mgvm/results/memory-traces'
date = "20260221-223953"
expanded_dir = f"{base_dir}/{date}/expanded"
metrics_dir = f"../pc-rdma-metrics"
trace_dir = f"{base_dir}/{date}"
print(f"{base_dir}/{date}")

/home/ehdgus7279/mgvm/results/memory-traces/20260221-223953


In [81]:
benchmark = 'gemm'
trace_file = f"{metrics_dir}/{benchmark}_pc_rdma_metrics.csv"

df = pd.read_csv(trace_file)
df

,interval,pc,ro_shared_cacheline_count,ro_shared_access_count,ro_shared_access_mean,ro_shared_access_std,ro_shared_shared_degree_mean,ro_shared_shared_degree_std,interval_total_cachelines_x,ro_shared_cacheline_ratio,...,private_access_mean,private_access_std,private_shared_degree_mean,private_shared_degree_std,interval_total_cachelines_y,private_cacheline_ratio,interval_total_access_y,private_access_ratio,pc_total_cachelines,pc_total_access
0,0.0,300012060,625.0,56288.0,90.060800,67.192673,12.812800,9.453759,179120.0,0.003489,...,5.729402,2.959488,0.0,0.0,179120.0,0.006912,487506.0,0.014550,1863,63381
1,0.0,300011776,1.0,288.0,288.000000,0.000000,35.000000,0.000000,179120.0,0.000006,...,0.000000,0.000000,0.0,0.0,NaN,0.000000,NaN,0.000000,1,288
2,0.0,300011796,1.0,288.0,288.000000,0.000000,35.000000,0.000000,179120.0,0.000006,...,0.000000,0.000000,0.0,0.0,NaN,0.000000,NaN,0.000000,1,288
3,0.0,300012064,116.0,139.0,1.198276,0.421588,3.439655,0.563973,179120.0,0.000648,...,1.000000,0.000000,0.0,0.0,179120.0,0.000073,487506.0,0.000027,129,152
4,0.0,300012056,42.0,66.0,1.571429,1.062506,20.500000,5.756227,179120.0,0.000234,...,1.166667,0.389249,0.0,0.0,179120.0,0.000067,487506.0,0.000029,54,80
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1807,1200000.0,300012068,3.0,3.0,1.000000,0.000000,1.000000,0.000000,1486.0,0.002019,...,0.000000,0.000000,0.0,0.0,NaN,0.000000,NaN,0.000000,3,3
1808,1200000.0,300012000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,NaN,0.000000,...,1.011264,0.105599,0.0,0.0,1486.0,0.537685,1593.0,0.507219,799,808
1809,1200000.0,300012004,0.0,0.0,0.000000,0.000000,0.000000,0.000000,NaN,0.000000,...,1.000000,0.000000,0.0,0.0,1486.0,0.084791,1593.0,0.079096,126,126
1810,1200000.0,300012016,0.0,0.0,0.000000,0.000000,0.000000,0.000000,NaN,0.000000,...,1.000000,0.000000,0.0,0.0,1486.0,0.000673,1593.0,0.000628,1,1


In [ ]:
df['interval']

In [83]:
df[['interval_total_access_x', 'interval_total_access_y']]

,interval_total_access_x,interval_total_access_y
0,487506.0,487506.0
1,487506.0,NaN
2,487506.0,NaN
3,487506.0,487506.0
4,487506.0,487506.0
...,...,...
1807,1593.0,NaN
1808,NaN,1593.0
1809,NaN,1593.0
1810,NaN,1593.0


In [57]:
df.sort_values(['interval', 'ro_shared_access_mean', 'rw_shared_access_mean'], ascending=[True, False, False]).reset_index()

,index,interval,pc,ro_shared_cacheline_count,ro_shared_access_count,ro_shared_access_mean,ro_shared_access_std,ro_shared_shared_degree_mean,ro_shared_shared_degree_std,interval_total_cachelines_x,...,private_access_mean,private_access_std,private_shared_degree_mean,private_shared_degree_std,interval_total_cachelines,private_cacheline_ratio,interval_total_access,private_access_ratio,pc_total_cachelines,pc_total_access
0,1,0.0,571330816,1.0,288.0,288.000000,0.000000,35.000000,0.000000,16015.0,...,0.000000,0.000000,0.0,0.0,NaN,0.000000,NaN,0.000000,1,288
1,2,0.0,571330836,1.0,272.0,272.000000,0.000000,33.000000,0.000000,16015.0,...,0.000000,0.000000,0.0,0.0,NaN,0.000000,NaN,0.000000,1,272
2,0,0.0,571330968,13388.0,17534.0,1.309680,0.462379,1.016657,0.127986,16015.0,...,1.329143,0.469991,0.0,0.0,16015.0,0.163909,21583.0,0.161655,16013,21023
3,4,10000.0,571331132,20.0,63.0,3.150000,0.366348,1.000000,0.000000,18096.0,...,4.251534,1.641828,0.0,0.0,18096.0,0.009008,25750.0,0.026913,183,756
4,3,10000.0,571330968,13332.0,17099.0,1.282553,0.450258,1.009451,0.096759,18096.0,...,0.000000,0.000000,0.0,0.0,NaN,0.000000,NaN,0.000000,14648,18871
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16578,16578,16490000.0,571331160,0.0,0.0,0.000000,0.000000,0.000000,0.000000,NaN,...,1.000000,0.000000,0.0,0.0,14890.0,0.000067,17257.0,0.000058,1,1
16579,16579,16490000.0,571331168,0.0,0.0,0.000000,0.000000,0.000000,0.000000,NaN,...,1.000000,0.000000,0.0,0.0,14890.0,0.000336,17257.0,0.000290,5,5
16580,16580,16490000.0,571331176,0.0,0.0,0.000000,0.000000,0.000000,0.000000,NaN,...,1.000000,0.000000,0.0,0.0,14890.0,0.000067,17257.0,0.000058,1,1
16581,16581,16490000.0,571331184,0.0,0.0,0.000000,0.000000,0.000000,0.000000,NaN,...,1.000000,0.000000,0.0,0.0,14890.0,0.000067,17257.0,0.000058,1,1


In [71]:
df['shared_cacheline_count'] = df['ro_shared_cacheline_count'] + df['rw_shared_cacheline_count']
df['shared_access_count'] =  df['ro_shared_access_count'] + df['rw_shared_access_count']
df['shared_access_mean'] =  df['shared_access_count'] / df['shared_cacheline_count']
df['shared_access_count_ratio'] =  df['shared_access_count'] / df['interval_total_access']
# df['shared_access_std'] =  df['ro_shared_access_std'] + df['rw_shared_access_std']
df['shared_chiplet_mean'] =  (df['ro_shared_shared_degree_mean'] + df['rw_shared_shared_degree_mean']) / 2

In [6]:
def fix_interval_total_access(df):
    """
    케이스 1: interval_total_access_x, interval_total_access_y 만 있음
    케이스 2: interval_total_access_x, interval_total_access_y, interval_total_access 모두 있음
    케이스 3: interval_total_access 만 있음 (이미 정상 → 아무것도 안 함)
    → 어떤 케이스든 최종적으로 interval_total_access 하나만 남김
    """
    has_x   = 'interval_total_access_x' in df.columns
    has_y   = 'interval_total_access_y' in df.columns
    has_raw = 'interval_total_access'   in df.columns

    if has_x and has_y:
        # _x, _y 중 NaN이 아닌 값으로 병합
        merged = df['interval_total_access_x'].combine_first(df['interval_total_access_y'])
        
        if has_raw:
            # 혹시 원본도 있다면 세 개를 모두 combine_first로 합쳐서 가장 완전한 값 사용
            merged = merged.combine_first(df['interval_total_access'])
        
        df['interval_total_access'] = merged
        df = df.drop(columns=['interval_total_access_x', 'interval_total_access_y'])
        print("✅ interval_total_access_x/_y 통합 완료")
    
    # 케이스 3: 이미 interval_total_access만 있으면 아무것도 안 함
    return df


In [ ]:
os.makedirs("./pc-rdma-metrics-fixed", exist_ok=True)

for benchmark in benchmarks:
    # if benchmark != 'syr2k':
        # continue
    if benchmark != 'gemm':
        continue

    df = pd.read_csv(f"../pc-rdma-metrics/{benchmark}_pc_rdma_metrics.csv", 
                        skipinitialspace=True)

    df = fix_interval_total_access(df)
    df.to_csv(f"./pc-rdma-metrics-fixed/{benchmark}_pc_rdma_metrics.csv")    